In [1]:
from config import CONVERSATION_EXAMPLES_PATH , OPENAI_API_KEY , PROMPTS_PATH, DICT_STATES, PREDEFINED_MESSAGES

# Import prompts path 
from config import *
from functions_auxiliary import *
# Import the vector store
from langchain.vectorstores import Chroma
# Import the embedding
from langchain.embeddings.openai import OpenAIEmbeddings 

from langchain.chat_models import ChatOpenAI

from langchain.memory import ConversationBufferMemory, ConversationSummaryMemory
from langchain.memory import ChatMessageHistory

from langchain.chains import ConversationChain

from langchain import PromptTemplate, LLMChain


from langchain.vectorstores.chroma import Chroma
from langchain.docstore.document import Document

from langchain.text_splitter import CharacterTextSplitter

In [2]:
def predict_new_summary(summarizer,memory):
    # Generate a summary of the conversation
    if summarizer.buffer == '':
        return ''
    else:
        last_messages = obtain_n_last_messages(memory)
        last_summary = summarizer.buffer()
        summarizer.buffer = sumarizer.predict_new_summary(messages= last_messages, existing_summary=last_summary)
    
def obtain_last_summary(summarizer):
    return summarizer.buffer()

def obtain_n_last_messages(memory, n_last_messages = 2):
    # If the number of messages is less than n_last_messages, we obtain all the messages
    if len(memory.messages) < n_last_messages:
        messages = memory.messages
    else: 
        # Obtain the last n messages
        messages = memory.messages[-n_last_messages:]
    return messages

def add_ai_messages_in_memory(message_content, memory):
    memory.add_ai_message(message_content)
    
def add_user_messages_in_memory(message_content, memory):
    memory.add_user_message(message_content)
    
def update_personal_information(personal_information_updater, previous_personal_information, new_patient_message):
    current_personal_information = personal_information_updater.predict(previous_information=previous_personal_information,
                                                                        new_messages= new_patient_message)
    return current_personal_information

In [3]:
def read_personal_information_template(path):
    personal_information_template = read_txt_files(path)
    return personal_information_template

In [4]:
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [5]:
# Creamos el archivo de memoria para poder obtener el resumen de la conversación
template_summary_conversation = read_txt_files(SUMMARY_CONVERSATION_PROMPT_PATH)
prompt_summary_conversation = PromptTemplate(input_variables=['summary', 'new_lines'], output_parser=None, partial_variables={}, 
                               template=template_summary_conversation, template_format='f-string', validate_template=True)
summarizer = ConversationSummaryMemory(llm=ChatOpenAI(model_name="gpt-3.5-turbo", temperature = 0.1), 
                                      prompt=prompt_summary_conversation,ai_prefix='Terapeutic assistant', human_prefix = 'Patient')

In [6]:
memory = ChatMessageHistory()

In [7]:
update_personal_information_prompt_txt = read_txt_files(UPDATE_PERSONAL_INFORMATION_PROMPT_PATH)
update_personal_information_prompt = PromptTemplate(input_variables=['previous_information', 'new_messages'], 
                                             output_parser=None, partial_variables={}, template=update_personal_information_prompt_txt, 
                                             template_format='f-string', validate_template=True)
personal_information_updater = LLMChain(llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature = 0.1), 
                                        verbose = False, prompt=update_personal_information_prompt)

In [8]:
template_first_session_master = read_txt_files(FIRST_SESSION_MASTER_PROMPT_PATH)
prompt_first_session_master = PromptTemplate(input_variables=['personal_information', 'previous_summary','patient_message'], 
                                             output_parser=None, partial_variables={}, template=template_first_session_master, 
                                             template_format='f-string', validate_template=True)
conversation = LLMChain(llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature = 0.5), 
                        verbose = False, prompt=prompt_first_session_master)

In [9]:
# Obtain patient message
patient_message = "Hola, soy nuevo en estas sesiones"
# Add patient message to memory
add_user_messages_in_memory(patient_message,memory)

# Predict a new summary
sumary = predict_new_summary(summarizer,memory)

# Update current personal information
previous_personal_information = read_txt_files(PERSONAL_INFORMATION_TEMPLATE_PATH)
current_personal_information = update_personal_information(personal_information_updater, previous_personal_information, patient_message)

# Predict AI assistant message
AI_asistant_message = conversation.predict(personal_information=current_personal_information, 
                                        previous_summary= sumary, 
                                        patient_message = patient_message)

# Add AI assistant message to memory
add_ai_messages_in_memory(AI_asistant_message,memory)

# Print AI assistant message
print(AI_asistant_message)


Hola, bienvenido/a a estas sesiones. Me alegra que hayas decidido comenzar tu proceso terapéutico. ¿Puedes decirme tu nombre para poder dirigirme a ti de manera adecuada? Además, ¿hay algo en particular que te haya llevado a buscar ayuda aquí? Estoy aquí para escucharte y brindarte el apoyo que necesites.


In [10]:
memory.messages

[HumanMessage(content='Hola, soy nuevo en estas sesiones', additional_kwargs={}, example=False),
 AIMessage(content='Hola, bienvenido/a a estas sesiones. Me alegra que hayas decidido comenzar tu proceso terapéutico. ¿Puedes decirme tu nombre para poder dirigirme a ti de manera adecuada? Además, ¿hay algo en particular que te haya llevado a buscar ayuda aquí? Estoy aquí para escucharte y brindarte el apoyo que necesites.', additional_kwargs={}, example=False)]